# Phan loai AAMI 3 lop (N/S/V) tren **Kaggle**  [v19]

Pipeline: 3 nhip lien ke -> denoiser (freeze) -> z-norm -> CNN -> SE -> pool -> noi RR(6)
-> head (classical MLP / quantum VQC + bypass) -> 3 lop. Inter-patient DS1/DS2.

**BAT BUOC trong Settings (panel phai):**
1. **Accelerator -> GPU** (P100 / T4)
2. **Internet -> On** (de tai PhysioNet + pip)

**Add Data (Upload -> New Dataset) 2 thu:**
1. Zip code (ten co chu **Denoise**) - ban v19
2. **denoise_runs_backup.zip** (chua run denoiser `*optimal*`)

Ket qua luu o `/kaggle/working/runs_cls`. Nho **Save Version** truoc khi dong de khong mat.

## 0. Kiem tra GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU! Settings -> Accelerator -> GPU'

## 1. Nap code tu /kaggle/input -> /kaggle/working/code

In [ ]:
import glob, os, shutil, zipfile

DEST = '/kaggle/working/code'
shutil.rmtree(DEST, ignore_errors=True)

hit = glob.glob('/kaggle/input/**/configs/default.yaml', recursive=True)
if hit:
    src_root = os.path.dirname(os.path.dirname(hit[0]))
    shutil.copytree(src_root, DEST)
    print('Nap code (da giai nen):', src_root)
else:
    zips = [z for z in glob.glob('/kaggle/input/**/*.zip', recursive=True)
            if 'Denoise' in os.path.basename(z) and 'runs_backup' not in os.path.basename(z)]
    assert zips, 'Khong thay zip code trong /kaggle/input (Add Data -> Upload).'
    shutil.rmtree('/kaggle/working/_unz', ignore_errors=True)
    os.makedirs('/kaggle/working/_unz', exist_ok=True)
    shutil.copy(max(zips, key=os.path.getsize), '/kaggle/working/_unz/code.zip')
    while True:
        zz = glob.glob('/kaggle/working/_unz/**/*.zip', recursive=True)
        if not zz:
            break
        for zp in zz:
            out = zp[:-4]; os.makedirs(out, exist_ok=True)
            try:
                with zipfile.ZipFile(zp) as z: z.extractall(out)
            except zipfile.BadZipFile: pass
            os.remove(zp)
    cfgp = glob.glob('/kaggle/working/_unz/**/configs/default.yaml', recursive=True)
    assert cfgp, 'Khong tim thay configs/default.yaml sau khi giai nen.'
    src_root = os.path.dirname(os.path.dirname(cfgp[0]))
    shutil.copytree(src_root, DEST)
    print('Nap code tu zip:', src_root)

os.chdir(DEST)
print('PROJECT DIR =', DEST, '| src/:', os.path.isdir('src'), '| configs/:', os.path.isdir('configs'))

## 2. Cai thu vien

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pennylane wfdb
import pennylane; print('pennylane', pennylane.__version__)

## 3. Tai du lieu PhysioNet (can Internet = On)

In [ ]:
import wfdb
wfdb.dl_database('mitdb', '/kaggle/working/data/MITDB')
wfdb.dl_database('nstdb', '/kaggle/working/data/NSTDB')
print('Downloaded.')

## 4. Sua config cho Kaggle

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['dataset_root']   = '/kaggle/working/data'
cfg['data']['processed_root'] = '/kaggle/working/data/processed'
cfg['output_root']            = '/kaggle/working/denoise_runs'
yaml.safe_dump(cfg, open('configs/default.yaml','w'), sort_keys=False)
c = cfg['classification']
print('processed_root =', cfg['data']['processed_root'])
print('classification :', {k: c[k] for k in ['n_classes','n_leads','n_context','z_norm']})

## 5. Khoi phuc denoiser *optimal* tu denoise_runs_backup.zip

In [ ]:
import glob, os, zipfile
os.makedirs('/kaggle/working/denoise_runs_restored', exist_ok=True)
bk = glob.glob('/kaggle/input/**/*runs_backup*.zip', recursive=True)
assert bk, 'Khong thay denoise_runs_backup.zip trong /kaggle/input.'
with zipfile.ZipFile(bk[0]) as z:
    z.extractall('/kaggle/working/denoise_runs_restored')
opt = [p for p in glob.glob('/kaggle/working/denoise_runs_restored/**/*optimal*', recursive=True)
       if os.path.isdir(p) and os.path.exists(os.path.join(p, 'checkpoint_best.pt'))]
assert opt, 'Khong tim thay run *optimal* co checkpoint_best.pt.'
DENOISE_RUN = opt[0]
print('DENOISE_RUN =', DENOISE_RUN)

## 6. Sinh du lieu phan loai (3 lop, context 3 nhip, co RR)

In [ ]:
!PYTHONPATH=. python -m src.prepare_classification_data --config configs/default.yaml
import pandas as pd
df = pd.read_csv('/kaggle/working/data/processed/manifest_cls.csv')
print('Tong beat:', len(df))
print(df.groupby(['split','label']).size())

## 7. Train
Kiem tra dong dau log phai co: `n_context=3 | augment=True | dropout=0.4`.

In [ ]:
# === E2. QUANTUM (VQC + bypass) + DENOISE  [model chinh] ===
import os
os.environ['DR'] = DENOISE_RUN
!PYTHONPATH=. python -m src.train_classifier --config configs/default.yaml \
    --denoise_run "$DR" --out_dir /kaggle/working/runs_cls/cls_quantum_denoise \
    --epochs 50 --batch_size 32 --lr 1e-3 --head quantum --encoding amplitude --pool_k 8

In [ ]:
# === E1. CLASSICAL + DENOISE (baseline) ===
import os
os.environ['DR'] = DENOISE_RUN
!PYTHONPATH=. python -m src.train_classifier --config configs/default.yaml \
    --denoise_run "$DR" --out_dir /kaggle/working/runs_cls/cls_classical_denoise \
    --epochs 50 --batch_size 64 --lr 1e-3 --head classical

In [ ]:
# === E3/E4 (ablation): tren beat NHIEU (khong khu nhieu) ===
!PYTHONPATH=. python -m src.train_classifier --config configs/default.yaml \
    --out_dir /kaggle/working/runs_cls/cls_quantum_noisy \
    --epochs 50 --batch_size 32 --head quantum --encoding amplitude --pool_k 8 --no_denoise
!PYTHONPATH=. python -m src.train_classifier --config configs/default.yaml \
    --out_dir /kaggle/working/runs_cls/cls_classical_noisy \
    --epochs 50 --batch_size 64 --head classical --no_denoise

## 8. Ket qua

In [ ]:
import json, os
for name in ['cls_quantum_denoise','cls_classical_denoise','cls_quantum_noisy','cls_classical_noisy']:
    p = f'/kaggle/working/runs_cls/{name}/metrics.json'
    if not os.path.exists(p):
        print(f'== {name} == chua co'); continue
    m = json.load(open(p)); t = m['test']
    print(f"== {name} ==  best_epoch={m['best_epoch']}  val_macroF1={round(m['val_macro_f1'],4)}")
    print("   acc=%.4f  weightedF1=%.4f  macroF1=%.4f" %
          (t['accuracy'], t.get('weighted_f1', float('nan')), t['macro_f1']))
    print("   F1 (N,S,V):", t['per_class_f1'])
    print("   Se        :", t['per_class_Se'])
    print("   +P        :", t['per_class_PP'])
    print("   confusion :", t['confusion'])

## 9. (Tuy chon) Cham nhanh checkpoint da luu - khong train lai
Dung khi phien bi ngat giua chung: `model_best.pt` van con trong run dir.

In [ ]:
import os
os.environ['DR'] = DENOISE_RUN
!PYTHONPATH=. python scripts/ensemble_cls.py --config configs/default.yaml \
    --denoise_run "$DR" --head quantum --encoding amplitude --pool_k 8 \
    --run_dirs /kaggle/working/runs_cls/cls_quantum_denoise

## 10. LUU KET QUA
`/kaggle/working` mat khi dong phien neu chua luu.
- **Save Version** (goc tren phai) -> luu toan bo output, dung lai lam Input lan sau.
- Hoac nen + tai ve bang cell duoi.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/runs_cls_backup', 'zip', '/kaggle/working/runs_cls')
print('Da nen: /kaggle/working/runs_cls_backup.zip -> tai o tab Output')